# LOBSTER Backtest — AAPL 2012-06-21

Analysis of the equity time series produced by `backtest_main` on the LOBSTER AAPL sample
(10-level L2, 400k messages). Prices are in LOBSTER scaled units (`$ * 10^4`), so PnL figures
are divided by 10,000 for dollar display.

Run the backtest first:
```bash
./build/backtest_main \
  --data-format lobster \
  --data data/lobster/AAPL/AAPL_2012-06-21_34200000_57600000_message_10.csv \
  --equity-csv benchmarks/backtest/lobster/equity.csv
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

EQUITY_CSV = '../benchmarks/backtest/lobster/equity.csv'
PRICE_SCALE = 10_000  # LOBSTER prices / pnl are dollars * 10^4

df = pd.read_csv(EQUITY_CSV)
# Convert wall_ns to pandas datetime for nicer plotting
df['time'] = pd.to_datetime(df['wall_ns'], unit='ns')
df['total_pnl_usd'] = df['total_pnl'] / PRICE_SCALE
df['real_pnl_usd']  = df['real_pnl']  / PRICE_SCALE
df['unreal_pnl_usd'] = df['unreal_pnl'] / PRICE_SCALE
print(f'Samples: {len(df)}')
print(f'Elapsed wall: {(df.wall_ns.max() - df.wall_ns.min())/1e9:.1f}s')
df.head()

In [ ]:
# Summary statistics.
peak = df['total_pnl_usd'].cummax()
drawdown = peak - df['total_pnl_usd']

# Crude "fill count" proxy: distinct non-zero volume steps.
fills = int((df['volume'].diff() > 0).sum())

summary = pd.Series({
    'samples':        len(df),
    'final_pnl_usd':  df['total_pnl_usd'].iloc[-1],
    'peak_pnl_usd':   df['total_pnl_usd'].max(),
    'max_drawdown_usd': drawdown.max(),
    'final_position': int(df['position'].iloc[-1]),
    'max_abs_position': int(df['position'].abs().max()),
    'final_volume':   int(df['volume'].iloc[-1]),
    'fills':          fills,
})
summary

In [ ]:
# Equity curve + position over time.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(df['time'], df['total_pnl_usd'], color='steelblue', linewidth=1.2, label='total PnL')
ax1.fill_between(df['time'], df['total_pnl_usd'], 0,
                 where=(df['total_pnl_usd'] >= 0), alpha=0.15, color='green')
ax1.fill_between(df['time'], df['total_pnl_usd'], 0,
                 where=(df['total_pnl_usd'] < 0), alpha=0.15, color='firebrick')
ax1.axhline(0, color='gray', linewidth=0.5)
ax1.set_ylabel('PnL ($)')
ax1.set_title('LiquidityTaker on LOBSTER AAPL 2012-06-21 — equity curve')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower left')

ax2.plot(df['time'], df['position'], color='black', linewidth=1.0, drawstyle='steps-post')
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.set_ylabel('Position (shares)')
ax2.set_xlabel('Wall-clock time (during backtest run)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../benchmarks/backtest/lobster', exist_ok=True)
plt.savefig('../benchmarks/backtest/lobster/equity_curve.png', dpi=120)
plt.show()

In [ ]:
# Underwater / drawdown plot.
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(df['time'], -drawdown, 0, color='firebrick', alpha=0.5)
ax.plot(df['time'], -drawdown, color='firebrick', linewidth=1)
ax.set_ylabel('Drawdown from peak ($)')
ax.set_title('Underwater curve')
ax.set_xlabel('Wall-clock time (during backtest run)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../benchmarks/backtest/lobster/drawdown.png', dpi=120)
plt.show()

## Interpretation

This run is the engine's smoke test, **not a trading result**:

- Pipeline runs end-to-end — LOBSTER event stream → MarketDataConsumer (replay mode) →
  MarketOrderBook → LiquidityTaker → OrderManager → FillSimulator → PositionKeeper →
  EquityRecorder. Strategy code is identical to what would ship live.
- Only a handful of fills — LiquidityTaker's default `threshold=0.5` and the FeatureEngine's
  reliance on order-flow imbalance mean the strategy rarely triggers on pure LOBSTER L2
  events. Tuning the threshold or moving to a MarketMaker + queue-aware fills (Phase 1.3)
  is the way to get meaningful PnL statistics from this dataset.
- For statistical metrics (Sharpe, drawdown, hit rate) that actually survive interview
  scrutiny, use the Binance aggTrades backtest in `backtest_binance.ipynb` — 3-6 months of
  data on a synthetic top-of-book is enough for a daily-return time series.